In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

import logging
logger = logging.getLogger(__name__)

In [ ]:
logging.basicConfig(format='%(asctime)s|%(levelname)s|%(message)s',
                    # filename='output.log', 
                    encoding='utf-8', 
                    level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
class ContrastiveLossEmbeddingModel(nn.Module):
    """
    Implements the ContrastiveLossEmbeddingModel.
    Implementation is nearly identical to TripletLossEmbeddingModel
    """
    def __init__(self, input_dim, embed_dim):
        super(ContrastiveLossEmbeddingModel, self).__init__()
        # Create the shared embedding layer
        self.embedding = nn.Linear(input_dim, embed_dim)

    def forward(self, x):
        # Compute embeddings
        return F.normalize(self.embedding(x), p=2, dim=1)

In [ ]:
class ContrastiveDataset(Dataset):
    """
    Implement contrastive dataset class.
    """
    def __init__(self, x1, x2, labels):
        assert x1.shape == x2.shape
        assert len(labels) == len(x1)
        self.num_samples, self.input_dim = x1.shape
        self.x1 = x1
        self.x2 = x2
        self.labels = labels
        
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return (
            self.x1[idx],
            self.x2[idx],
            self.labels[idx]
            # torch.tensor(self.x1[idx].detach().clone(), dtype=torch.float32),
            # torch.tensor(self.x2[idx].detach().clone(), dtype=torch.float32),
            # torch.tensor(self.labels[idx].detach().clone(), dtype=torch.float32)
        )

In [ ]:
class ContrastiveLoss(nn.Module):
    """
    Because PyTorch does not have a built-in contrastive loss function, we create one.
    """
    def __init__(self, margin=1.0, dist_func=None):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

        if dist_func is None:
            self.dist_func = F.pairwise_distance
        else:
            self.dist_func = dist_func

    def forward(self, output1, output2, label):
        distances = self.dist_func(output1, output2)
        loss = torch.mean(label * distances.pow(2) +
                          (1 - label) * F.relu(self.margin - distances).pow(2))
        return loss

In [ ]:
def cosine_distance(x1, x2):
    """
    Implement a cosine_distance function as PyTorch does not have a built-in one.
    """
    x1 = F.normalize(x1, p=2, dim=1)
    x2 = F.normalize(x2, p=2, dim=1)
    cosine_sim = torch.sum(x1 * x2, dim=1)
    return 1 - cosine_sim  # convert similarity to distance


def train_contrastive_loss(dataset, embed_dim,
                       dist_func=None,
                       margin=1.0,
                       batch_size=256,
                       epochs=10,
                       lr=0.001
                       ):
    """
    Training loop for triplet margin loss.

    Args:
        dataset (ContrastiveLossDataset) : The training dataset.
        embed_dim (int) : The size of the embedding vectors.
        dist_func (callable) : If using custom distance function (default is Euclidean distance).
        margin (float) : The size of the margin learned (larger forces more separation between positive and negative).
        batch_size (int) : Training batch size.
        epochs (int) : Training epochs.
        lr (float) : The learning rate.

    Returns:
        model (ContrastiveLossEmbeddingModel) : The trained model.
        loss (np.array(float)) : The training loss for each epoch.
    """

    input_dim = dataset.input_dim
    model = ContrastiveLossEmbeddingModel(input_dim, embed_dim)

    # Create loss function module
    criterion = ContrastiveLoss(margin, dist_func)

    # Init optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # Training loop
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    loss_history = np.zeros(epochs)
    for i_e, epoch in enumerate(range(epochs)):
        total_loss = 0
        for x1, x2, labels in dataloader:
            optimizer.zero_grad()
            # Forward pass
            x1_embed = model(x1)
            x2_embed = model(x2)
            # Compute loss
            loss = criterion(x1_embed, x2_embed, labels)
            # Backpropagation
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        loss_history[i_e] = total_loss/len(dataloader)
        
        logging.debug(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader):.4f}")

    return model, loss_history
    

In [ ]:

def vectorize(series):
    """
    Takes a pandas Series of embedding_json strings
    and returns a NumPy array of shape (N, 384).
    """
    # Ensure string
    s = series.astype(str).str.strip()

    # Remove brackets, split into float lists
    vecs = (
        s.str[1:-1]                   # strip '[' and ']'
         .str.split(',')              # split by comma
         .apply(lambda x: np.array(x, dtype=np.float32))
         .tolist()
    )

    # Stack into (n_samples, embed_dim)
    return np.vstack(vecs)

In [ ]:
global_db = pd.read_csv('/data/elugos/global_db_111125.csv')
df_train = pd.read_csv(Path('/data/elugos/event_embedding/train_20251214.csv'))

In [ ]:
def clean_sample(df,global_db):
    ## get longest sbert text
    df["text_len"] = df["sbert_text"].str.len()
    df_clean = df.sort_values("text_len", ascending=False).drop_duplicates(subset="GlobalEventID", keep="first")
    df_clean = df_clean.drop(columns=["text_len"])
    ## get all rows that exist in df from global_db and clean to make sure they match, drop all other rows
    ids = df_clean["GlobalEventID"]
    global_play_db = global_db[global_db["GlobalEventID"].isin(ids)].copy()
    valid_ids = set(global_play_db["GlobalEventID"])
    df_clean = df_clean[df_clean["GlobalEventID"].isin(valid_ids)].copy()
    ## add globaldb to df_clean
    df_clean = df_clean.merge(
    global_play_db,
    on="GlobalEventID",
    how="left",
    suffixes=("", "_db")
    )

    return df_clean,global_play_db

In [ ]:
df_train_sample,global_play_db = clean_sample(df_train,global_db)
df_train_sample = df_train_sample.dropna(subset=["embedding_json_title"])
print(f"Training samples: {len(df_train_sample)}")
df_train_sample.columns


In [ ]:
cameo_codes = {
    "MAKE PUBLIC STATEMENT": "1",
    "APPEAL": "2",
    "EXPRESS INTENT TO COOPERATE": "3",
    "CONSULT": "4",
    "ENGAGE IN DIPLOMATIC COOPERATION": "5",
    "ENGAGE IN MATERIAL COOPERATION": "6",
    "PROVIDE AID": "7",
    "YIELD": "8",
    "INVESTIGATE": "9",
    "DEMAND": "10",
    "DISAPPROVE": "11",
    "REJECT": "12",
    "THREATEN": "13",
    "PROTEST": "14",
    "EXHIBIT FORCE POSTURE": "15",
    "COERCE": "16",
    "ASSAULT": "17",
    "FIGHT": "19",
    "USE UNCONVENTIONAL MASS VIOLENCE": "20",
}

In [ ]:

# def sample_filtered(first_item, df: pd.DataFrame, n: int=1, same_group: bool=True):
#     """
#     Generate a sample for a first_item.

#     Parameters:
#     first_item : pd.DataFrame
#         The sampled item.
#     df : pd.DataFrame
#         The df to filter.
#     n : int
#         How many items to sample.
#     same_group : bool
#         If sampling from the same group or different.

#     Returns
#     pd.DataFrame
#         A dataframe with the sample
#     """
#     if same_group:
#         df_filtered = df[df['EventLabel']==first_item.EventLabel]
#     else:
#         df_filtered = df[df['EventLabel']!=first_item.EventLabel]
    
#     return df_filtered.sample(n=n)

def sample_group(g: pd.DataFrame, df: pd.DataFrame, same: bool=True):
    """
    Generates a sample for a group of EventLabels
    """
    event_label = list(g['EventLabel'].unique())[0]
    if same:
        df_sub = df[df['EventLabel'] == event_label]
    else:
        df_sub = df[df['EventLabel'] != event_label]
    return df_sub.sample(n=len(g), replace=True)


def make_sampler(df: pd.DataFrame, num: int, col: str='embedding_json_title', balanced: bool=False):
    """
    Generate a set of training samples from a DataFrame.

    This function selects `num` samples from the specified column of the DataFrame,
    optionally balancing the selection across classes or categories.

    Parameters
    ----------
    df : pandas.DataFrame
        The input DataFrame containing the data.
    num : int
        The total number of samples to generate.
    col : str, optional
        Name of the column containing feature embeddings or text (default: 'embedding_json_title').
    balanced : bool, optional
        If True, attempt to balance samples across classes or categories (default: True).

    Returns
    -------
    pandas.DataFrame
        A DataFrame containing the sampled rows.
    """
    df['EventLabel'] = df['EventCode'].astype(str).str.split("_").str[-1]

    if not balanced:
        # sample i and j independently
        # This may (albeit unlikely) generate samples where i==j, which is fine, since that would imply the events are the same.
        # Alternatively, we can safely drop those samples, if needed.
        i = df.sample(n=num, random_state=42, replace=True)
        j = df.sample(n=num, random_state=1, replace=True)
    else:
        # Sample a balanced number of pairs
        # Sample positive pairs
        sample_pos = df.sample(n=num//2, random_state=0, replace=True)

        i_pos = list()
        j_pos = list()
        groups_pos = sample_pos.groupby('EventLabel')
        for name, g in groups_pos:
            i_pos.append(g)
            j_pos.append(sample_group(g, df, True))
        j_pos = pd.concat(j_pos).reset_index()
        i_pos = pd.concat(i_pos).reset_index()

        assert (i_pos['EventLabel']!=j_pos['EventLabel']).sum() == 0
        sample_neg = df.sample(n=num//2, random_state=13, replace=True)
        i_neg = list()
        j_neg = list()
        groups_neg = sample_neg.groupby('EventLabel')
        for name, g in groups_neg:
            i_neg.append(g)
            j_neg.append(sample_group(g, df, False))
        i_neg = pd.concat(i_neg).reset_index()
        j_neg = pd.concat(j_neg).reset_index()

        assert (i_neg['EventLabel']==j_neg['EventLabel']).sum() == 0

        # j_pos = list()
        # for item in i_pos.itertuples():
        #     j_pos.append(sample_filtered(item, df, 1, True))  # Sample 1 positive (same group) for each element in i_pos
        # j_pos = pd.concat(j_pos).reset_index()

        # i_neg = df.sample(n=num//2, random_state=7, replace=True)
        # j_neg = list()
        # for item in i_neg.itertuples():
        #     j_neg.append(sample_filtered(item, df, 1, False))  # Sample 1 positive (same group) for each element in i_pos
        # j_neg = pd.concat(j_neg).reset_index()

        i = pd.concat([i_pos, i_neg])
        j = pd.concat([j_pos, j_neg])
        

    # reset index so pairs line up
    i = i.reset_index(drop=True)
    j = j.reset_index(drop=True)

    # # Extract only the FIRST part of EventCode (before underscore)
    # i_event_major = i["EventCode"].astype(str).str.split("_").str[-1]
    # j_event_major = j["EventCode"].astype(str).str.split("_").str[-1]

    y = (i['EventLabel'].values == j['EventLabel'].values).astype(int)


    # Build sampler
    sampler = pd.DataFrame({
        "GlobalEventID-i": i["GlobalEventID"],
        "GlobalEventID-j": j["GlobalEventID"],
        "features-i": i[col],
        "features-j": j[col],
        "EventCode-i": i['EventLabel'],
        "EventCode-j": j['EventLabel'],
        "y": y
    })

    return sampler




In [ ]:
n_training = 100000
sampler = make_sampler(df_train_sample,n_training, balanced=True)

print(f"Input training data: {len(df_train_sample)} articles.")
print(f"Training pairs: {len(sampler)}")
print(f"  {sampler.value_counts('y')}")

In [ ]:


i_np = vectorize(sampler["features-i"])
j_np = vectorize(sampler["features-j"])

i = torch.from_numpy(i_np)
j = torch.from_numpy(j_np)
# Random labels: 0 if dissimilar, 1 if similar.
#make the labels if similar 1, 0 if not
labels = torch.tensor(sampler["y"].values, dtype=torch.float32)

logger.info(f"{i.shape}, {j.shape}, {labels.shape}")


In [ ]:
input_dim = 384  # Just like SBERT embeddings
embed_dim = 300  # Chosen embedding dimension
margin = 1.0  # Choose a margin
batch_size=512  # the batch size
n_epochs = 30  # the number of epochs
lr = 0.1  # the learning rate

dataset = ContrastiveDataset(i, j, labels)
model, loss = train_contrastive_loss(dataset, embed_dim, margin=margin,
                                     dist_func=cosine_distance, 
                                     batch_size=batch_size,
                                     epochs=n_epochs,
                                     lr=lr
                                     )
logger.info(f"Model trained!")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(loss, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Contrastive Loss Over Epochs")
plt.grid(True)
plt.show()

In [ ]:
# merge metadata for i
meta_i = sampler[["GlobalEventID-i"]].merge(
    global_db[["GlobalEventID", "EventCode"]],
    left_on="GlobalEventID-i",
    right_on="GlobalEventID",
    how="left"
).drop(columns=["GlobalEventID"]).rename(columns={"EventCode": "EventCode-i"})

# merge metadata for j
meta_j = sampler[["GlobalEventID-j"]].merge(
    global_db[["GlobalEventID", "EventCode"]],
    left_on="GlobalEventID-j",
    right_on="GlobalEventID",
    how="left"
).drop(columns=["GlobalEventID"]).rename(columns={"EventCode": "EventCode-j"})


In [ ]:
def avg_pairwise_distance_by_label(x1, x2, labels, metric="cosine"):
    """
    Works with BOTH numpy arrays and torch tensors.
    Computes avg distance for same-label and opposite-label pairs.
    """

    # --- Convert everything to torch tensors if needed ---
    if isinstance(x1, np.ndarray):
        x1 = torch.from_numpy(x1)
    if isinstance(x2, np.ndarray):
        x2 = torch.from_numpy(x2)
    if isinstance(labels, np.ndarray):
        labels = torch.from_numpy(labels)

    x1 = x1.float()
    x2 = x2.float()
    labels = labels.int()

    # --- Distance computation ---
    if metric == "cosine":
        x1n = F.normalize(x1, p=2, dim=1)
        x2n = F.normalize(x2, p=2, dim=1)
        distances = 1 - torch.sum(x1n * x2n, dim=1)

    elif metric == "euclidean":
        distances = torch.norm(x1 - x2, dim=1)

    else:
        raise ValueError("metric must be 'cosine' or 'euclidean'")

    distances = distances.cpu().numpy()
    labels = labels.cpu().numpy()

    same_mask = labels == 1
    opp_mask  = labels == 0

    same_dist = distances[same_mask]
    opp_dist = distances[opp_mask]

    avg_same = distances[same_mask].mean() if same_mask.any() else np.nan
    avg_opp  = distances[opp_mask].mean() if opp_mask.any() else np.nan

    return same_dist, opp_dist,{
        "avg_same": float(avg_same),
        "avg_opposite": float(avg_opp),
        "n_same": int(same_mask.sum()),
        "n_opposite": int(opp_mask.sum())
    }

In [ ]:
model.eval()

with torch.no_grad():
    i_embed = model(i).cpu().numpy()   # shape: (N, embed_dim)
    j_embed = model(j).cpu().numpy()   # shape: (N, embed_dim)
    
all_embeddings = np.vstack([i_embed, j_embed])

global_ids = np.concatenate([sampler["GlobalEventID-i"].values,
                             sampler["GlobalEventID-j"].values])

eventcodes = np.concatenate([
    meta_i["EventCode-i"].astype(str).str.rsplit("_", n=1).str[-1].values,
    meta_j["EventCode-j"].astype(str).str.rsplit("_", n=1).str[-1].values
])


all_labels = np.concatenate([labels.numpy(), labels.numpy()])
pair_flag   = np.array([0]*len(i_embed) + [1]*len(j_embed))  # blue = i, orange = j


In [ ]:
sbert_same_dist, sbert_opp_dist, stats_sbert_35000 = avg_pairwise_distance_by_label(i, j, labels, metric="cosine")

# Learned embeddings
learned_same_dist, learn_opp_dist, stats_learned_35000 = avg_pairwise_distance_by_label(i_embed, j_embed, labels, metric="cosine")

In [ ]:
stats_learned_35000

In [ ]:
stats_sbert_35000

In [ ]:
# Sample data for testing

# Get event ids used in training and remove them from the test
training_event_ids = set(list(sampler['GlobalEventID-i'].unique()) + list(sampler['GlobalEventID-j']))
df_test_sample = df_train_sample[~df_train_sample['GlobalEventID'].isin(training_event_ids)].copy()
print(f"Found {len(df_test_sample)} samples not seen in training.")

sample_test = make_sampler(df_test_sample, 1000)

# merge metadata for i
meta_i = sample_test[["GlobalEventID-i"]].merge(
    global_db[["GlobalEventID", "EventCode"]],
    left_on="GlobalEventID-i",
    right_on="GlobalEventID",
    how="left"
).drop(columns=["GlobalEventID"]).rename(columns={"EventCode": "EventCode-i"})

# merge metadata for j
meta_j = sample_test[["GlobalEventID-j"]].merge(
    global_db[["GlobalEventID", "EventCode"]],
    left_on="GlobalEventID-j",
    right_on="GlobalEventID",
    how="left"
).drop(columns=["GlobalEventID"]).rename(columns={"EventCode": "EventCode-j"})

In [ ]:
i_np = vectorize(sample_test["features-i"])
j_np = vectorize(sample_test["features-j"])

i_test = torch.from_numpy(i_np)
j_test = torch.from_numpy(j_np)
# Random labels: 0 if dissimilar, 1 if similar.
#make the labels if similar 1, 0 if not
labels_test = torch.tensor(sample_test["y"].values, dtype=torch.float32)

logger.info(f"{i_test.shape}, {j_test.shape}, {labels_test.shape}")

In [ ]:
model.eval()

with torch.no_grad():
    i_embed_test = model(i_test).cpu().numpy()   # shape: (N, embed_dim)
    j_embed_test = model(j_test).cpu().numpy()   # shape: (N, embed_dim)
    
all_embeddings = np.vstack([i_embed_test, j_embed_test])

global_ids = np.concatenate([sample_test["GlobalEventID-i"].values,
                             sample_test["GlobalEventID-j"].values])

eventcodes = np.concatenate([
    meta_i["EventCode-i"].astype(str).str.rsplit("_", n=1).str[-1].values,
    meta_j["EventCode-j"].astype(str).str.rsplit("_", n=1).str[-1].values
])


all_labels = np.concatenate([labels_test.numpy(), labels_test.numpy()])
pair_flag   = np.array([0]*len(i_embed_test) + [1]*len(j_embed_test))  # blue = i, orange = j

In [ ]:
test_sbert_same_dist, test_sbert_opp_dist, test_stats_sbert_35000 = avg_pairwise_distance_by_label(i_test, j_test, labels_test, metric="cosine")

# Learned embeddings
test_learned_same_dist, test_learn_opp_dist, test_stats_learned_35000 = avg_pairwise_distance_by_label(i_embed_test, j_embed_test, labels_test, metric="cosine")

In [ ]:
print(test_stats_learned_35000)

In [ ]:
#np.savetxt('sbert_same_euc_64_1.csv', sbert_same_dist, delimiter=',')
#np.savetxt('sbert_opp_euc_64_1.csv', sbert_opp_dist, delimiter=',')
np.savetxt('learned_same_cos_64_1.csv', learned_same_dist, delimiter=',')
np.savetxt('learned_opp_cos_64_1.csv', learn_opp_dist, delimiter=',')

np.savetxt('test_learned_same_cos_64_1.csv', test_learned_same_dist, delimiter=',')
np.savetxt('test_learned_opp_cos_64_1.csv', test_learn_opp_dist, delimiter=',')

In [ ]:
from datetime import datetime

torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim": input_dim,
    "embed_dim": embed_dim,
    "margin": margin,
    "epochs": n_epochs,
    "date": datetime.now().strftime("%Y%m%dT%H%M%S")
}, "contrastive_64_1.pt")

In [ ]:
emb2d_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(all_embeddings)

emb2d_pca = PCA(n_components=2).fit_transform(all_embeddings)


In [ ]:
df_plot = pd.DataFrame({
    "x": emb2d_tsne[:,0],
    "y": emb2d_tsne[:,1],
    "GlobalEventID": global_ids,
    
    "EventCode": eventcodes.astype(int)
})

fig = px.scatter(
    df_plot,
    x="x",
    y="y",
    color="EventCode",
    hover_data=["GlobalEventID", "EventCode"],
    title="Learned Embeddings Colored by EventCode"
)

fig.update_traces(marker=dict(size=6, opacity=0.85))
fig.update_layout(width=900, height=600)
fig.show()

In [ ]:
df_plot = pd.DataFrame({
    "x": emb2d_pca[:,0],
    "y": emb2d_pca[:,1],
    "GlobalEventID": global_ids,
    
    "EventCode": eventcodes.astype(int)
})

fig = px.scatter(
    df_plot,
    x="x",
    y="y",
    color="EventCode",
    hover_data=["GlobalEventID", "EventCode"],
    title="Learned Embeddings Colored by EventCode"
)

fig.update_traces(marker=dict(size=6, opacity=0.85))
fig.update_layout(width=900, height=600)
fig.show()